# 03. Feature Research — 중복 Feature, 집계 Feature, Exposure 집중도

> 이 노트북은 **설명용**이다. 계산 로직은 모두 `src/credit_risk/eda.py`, `src/credit_risk/stats.py`에 있으며, 보고서는 `python -m credit_risk.cli eda`로 재현한다.
>
> 용어: TARGET 평균 = **관측 상환곤란율 (Observed Difficulty Rate)**, AMT_CREDIT = **Exposure Proxy (AMT_CREDIT)**. 모든 관계는 연관이며 인과가 아니다. 합성 데이터가 적재된 경우 결과는 해석에 사용하지 않는다.

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("DATA_DIR", "./data/raw")  # only used for the data directory; DB creds come from .env

import pandas as pd
from credit_risk.config import get_settings
from credit_risk.db import connect
from credit_risk import eda, stats

settings = get_settings()
pd.set_option("display.max_columns", 50)

## 상관관계 / 중복 Feature (|r| > 0.9)
mart 수치형 컬럼의 md5 정렬 결정적 표본을 사용한다.

In [ ]:
with connect(settings) as conn:
    meta = eda.table_columns(conn, 'analytics', 'customer_feature_mart')
    num_cols = meta.loc[meta['data_type'].isin(['double precision', 'bigint', 'integer', 'smallint', 'numeric']), 'column_name']
    num_cols = [c for c in num_cols if c not in ('sk_id_curr', 'target')]
    sample = eda.load_numeric_sample(conn, num_cols, eda.CORR_SAMPLE)
corr, pairs = eda.correlation_check(sample)
pairs.head(30)

## 다중공선성 참고 — VIF

In [ ]:
eda.vif_table(sample, ['amt_income_total', 'amt_credit', 'amt_annuity', 'amt_goods_price',
                       'credit_income_ratio', 'annuity_income_ratio', 'ext_source_mean'])

## 보조 테이블 집계 Feature vs TARGET
이력 보유 고객(non-NULL) 내 비교 + 이력 부재는 별도 구간.

In [ ]:
with connect(settings) as conn:
    df = eda.load_train_frame(conn, ['target', 'amt_credit', 'inst_late_ratio', 'prev_refused_ratio',
                                     'bur_bb_worst_status', 'cc_utilization_mean', 'ext_source_mean', 'amt_income_total'])
eda.target_comparison(df, ['inst_late_ratio', 'prev_refused_ratio', 'cc_utilization_mean'], 'aggregate')[
    ['feature', 'coverage', 'median1', 'median0', 'effect_size', 'effect_label']]

In [ ]:
eda.binned_risk(df, eda.ratio_band(df['inst_late_ratio'], [0.05, 0.2]), 'inst_late_ratio_band', 'inst_late_ratio')['table']

## Exposure Proxy (AMT_CREDIT) 집중도
관측 상환곤란율이 전체의 1.2배 이상인 세그먼트의 고객 비중 vs Exposure 비중.

In [ ]:
seg = settings.project['segments']
band = eda.income_band(df['amt_income_total'], seg['income_bins'], seg['income_labels'])
rt = stats.rate_table(band, df['target'], df['amt_credit'])
overall = df['target'].mean()
eda.exposure_concentration(rt, overall, min_n=30), eda.score_tail_concentration(df)